In [ ]:
import os
import io
import argparse
import shutil
import glob
from google.cloud import storage
from google import genai
import pandas as pd
import fitz # PyMuPDF package 


os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "C:/Users/718/Desktop/llm-service-account.json"
# Generate the inputs arguments parser
parser = argparse.ArgumentParser(description="Command description.")

gcp_project = "apcomp215-group88"
bucket_name = "group88-bucket-1"
EMBEDDING_MODEL = "text-embedding-004"
storage_client = storage.Client()
bucket = storage_client.bucket(bucket_name)
llm_client = genai.Client(vertexai=True, project=gcp_project, location="us-central1")

In [8]:
prompt = """
    You are a medical safety analyst.

    Please generate 10 examples of medical safety incidents across different departments,
    including internal medicine, surgery, ob/gyn/nicu, radiology/imaging, and outpatient/ER.

    Each event description should be NO LESS THAN 60 words, detailed and realistic, either personal or based on real historical public cases.  Need to describe what hospital did lead to the incidents and the reason behind the story.
    
    Do not label or classify them with severity levels (e.g., SSE/PSE/NME).
    Focus on medically plausible situations reflecting real hospital or clinical settings.

    Format the output in a structured, numbered list like this:

    1. [Department Name]: [Detailed incident description...]
    2. [Department Name]: [Detailed incident description...]
    ...
    100. [Department Name]: [Detailed incident description...]

    SSE Example:
    A patient was admitted to the critical care unit with congestive heart failure and later has a new complaint of chest pain persisting over several hours. Tylenol is administered but does not decrease the patient's pain scale rating. The Resident orders a laboratory work-up. An EKG shows that the patient is experiencing an acute ST-elevation myocardial infarction. The attending cardiologist is called, but does not respond to multiple pages. The nurse does not escalate the patient's emergent condition to other physicians or the rapid response team. The patient continues to decompensate, codes and expires.

    PSE Example:
    A patient was being treated in the Cardiac Progressive Care Unit and had a heparin lock in place. As part of the routine hep-lock care, the nurse administered what she thought was the usual hep-lock flush solution from an unlabeled syringe she previously placed at the bedside. However, the syringe actually contained Cardizem, which she administered to the patient. The patient experienced an immediate hypotensive reaction, which quickly resolved without further incident.

    NME Example:
    A nurse in a health care systems' float pool is assigned to work at the organization's Assisted Living Facility for several days. He begins passing out medications to the residents and some of them aren't in their room but in the facility's Day Room. Most of the resident's don't wear arm bands but have pictures inside their rooms for identification. The nurse depended on patient's selfidentifying when passing meds in the Day Room. He had been introduced to the patients earlier that day and thinking he knew the patients, did the identification with saying the patient's name as it seemed quicker and he was behind in this task. When the patient agreed, he gave the pill packet to them. Another resident hearing what transpired walked up to tell the nurse that this patient was responding to the wrong name. The nurse took the medications back and began to perform the identification process correctly by asking the patient to state name and DOB.

    NSE Example:
    A healthy Gravida 2 Para1 40-week gestation mother presents in active labor requiring routine monitoring according to Labor & Delivery protocols. Labor progresses to delivery of a healthy 6-pound infant with Apgar scores of 10 and 10. Within 2 minutes post-delivery, the mother, who is still being monitored, begins to develop extreme respiratory distress and signs of disseminated intravascular clotting (DIC) syndrome. Despite all emergent efforts to reverse the situation, the mother succumbs to what is later diagnosed as an amniotic fluid embolism.
    """

In [9]:

from vertexai.generative_models import GenerativeModel, GenerationConfig
from google.genai import types
response = llm_client.models.generate_content(
    model="gemini-2.5-flash", contents=prompt,
    config=types.GenerateContentConfig(
        temperature=0.7,
        max_output_tokens=8000
    )
)
paragraph = response.text

print("Generated text:")
print(paragraph)


Generated text:
Here are 10 examples of medical safety incidents across different departments:

1.  **Internal Medicine**: A 72-year-old patient with type 2 diabetes and renal impairment was admitted to the internal medicine ward for pneumonia. The physician ordered 10 units of Novolog insulin to be given subcutaneously before meals. However, during the evening medication pass, the nurse mistakenly administered 100 units of Novolog, confusing the order for "10 units" with a "100 unit" syringe due to poor lighting and a busy shift. The patient became severely hypoglycemic, requiring immediate dextrose infusion and close monitoring in the ICU for 24 hours. Contributing factors included high workload for the nurse, dim lighting in the medication room, similar packaging of different insulin concentrations, and a lack of independent double-check for high-alert medications like insulin during a particularly chaotic shift on an understaffed ward.

2.  **Surgery**: During an abdominal hysterec

In [17]:
import pandas as pd
import re
pattern = r"\*\*(.*?)\*\*: (.*?)\n"
matches = re.findall(pattern, paragraph)
df = pd.DataFrame(matches, columns=["Department", "Incident Description"])
os.makedirs("Data", exist_ok=True)
df.to_excel("Data/medical_incidents.xlsx", index=False)

C:\Users\718\AppData\Local\Temp/ipykernel_17008/1218193490.py:7: UserWarning: Pandas requires version '3.0.5' or newer of 'xlsxwriter' (version '3.0.1' currently installed).
  df.to_excel("Data/medical_incidents.xlsx", index=False)


In [11]:
with open("medical_incidents_v2.txt", "w", encoding="utf-8") as f:
    f.write(paragraph)

In [2]:
def generate_text_embeddings(chunks, dimensionality: int = 256, batch_size=250, max_retries=5, retry_delay=5):
    # Max batch size is 250 for Vertex AI
    all_embeddings = []

    for i in range(0, len(chunks), batch_size):
        batch = chunks[i:i+batch_size]

        # Retry logic with exponential backoff
        retry_count = 0
        while retry_count <= max_retries:
            try:
                response = llm_client.models.embed_content(
                    model=EMBEDDING_MODEL,
                    contents=batch,
                    config=types.EmbedContentConfig(
                        output_dimensionality=dimensionality),
                )
                all_embeddings.extend(
                    [embedding.values for embedding in response.embeddings])
                break

            except errors.APIError as e:
                retry_count += 1
                if retry_count > max_retries:
                    print(
                        f"Failed to generate embeddings after {max_retries} attempts. Last error: {str(e)}")
                    raise

                # Calculate delay with exponential backoff
                wait_time = retry_delay * (2 ** (retry_count - 1))
                print(
                    f"API error (code: {e.code}): {e.message}. Retrying in {wait_time} seconds (attempt {retry_count}/{max_retries})...")
                time.sleep(wait_time)

    return all_embeddings

In [7]:
from pypdf import PdfReader
from io import BytesIO
from pypdf import PdfReader
from semantic_splitter import SemanticChunker

# Vertex AI
from google import genai
from google.genai import types
from google.genai.types import Content, Part, GenerationConfig, ToolConfig
from google.genai import errors


storage_client = storage.Client()
source_blob_name = "RAG_taxtbook/hpi-sec-sser.pdf" 

bucket = storage_client.bucket(bucket_name)
blob = bucket.blob(source_blob_name)
pdf_bytes = blob.download_as_bytes()
pdf_stream = BytesIO(pdf_bytes)

doc = fitz.open(stream=pdf_stream, filetype="pdf")

pages = []
for page in doc:
    text = page.get_text("text")   # preserves line breaks
    # remove headers and footers
    clean_text = "\n".join([
        line for line in text.splitlines()
        if not line.strip().startswith("WHITE PAPER") and
           not line.strip().startswith("©Press Ganey")
    ])
    pages.append(clean_text.strip())

full_text = "\n\n".join(pages)
print("✅ Extracted characters:", len(full_text))

✅ Extracted characters: 178499


In [7]:
with open("extracted_text.txt", "w", encoding="utf-8") as f:
    f.write(full_text)

In [8]:
print("Performing semantic chunking...")
text_splitter = SemanticChunker(
    embedding_function=generate_text_embeddings,
    breakpoint_threshold_type="percentile",  
    breakpoint_threshold_amount=60           # lower = more chunks
)

text_chunks = text_splitter.create_documents([full_text])
text_chunks = [doc.page_content for doc in text_chunks]
print("✅ Number of semantic chunks:", len(text_chunks))

Performing semantic chunking...
✅ Number of semantic chunks: 469


In [9]:
OUTPUT_FOLDER = "outputs"
chunk_file = "semantic chunks-hpi-sec-sser.pdf.jsonl"
if text_chunks is not None:
    # Save the chunks
    data_df = pd.DataFrame(text_chunks, columns=["chunk"])
    data_df["book"] = "hpi-sec-sser.pdf"
    print("Shape:", data_df.shape)
    print(data_df.head())

    jsonl_filename = os.path.join(
        OUTPUT_FOLDER, chunk_file)
    os.makedirs(OUTPUT_FOLDER, exist_ok=True)
    with open(jsonl_filename, "w") as json_file:
        json_file.write(data_df.to_json(orient='records', lines=True))

Shape: (469, 2)
                                               chunk              book
0  1\nHPI SEC & \nSSER\nPatient safety measuremen...  hpi-sec-sser.pdf
1  While healthcare holds healing without \nharm ...  hpi-sec-sser.pdf
2  It serves as the foundation for \nthe calculat...  hpi-sec-sser.pdf
3  •\t\nFocus on other frequently \nencountered i...  hpi-sec-sser.pdf
4  The application of these concepts has \nbeen e...  hpi-sec-sser.pdf


In [10]:

GCP_LOCATION = "us-central1"
EMBEDDING_MODEL = "text-embedding-004"
EMBEDDING_DIMENSION = 256
GENERATIVE_MODEL = "gemini-2.0-flash-001"
INPUT_FOLDER = "input-datasets"
OUTPUT_FOLDER = "outputs"
CHROMADB_HOST = "llm-rag-chromadb"
CHROMADB_PORT = 8000
import time


# Get the list of chunk files
jsonl_files = glob.glob(os.path.join(
    OUTPUT_FOLDER, chunk_file))
print("Number of files to process:", len(jsonl_files))

# Process
for jsonl_file in jsonl_files:
    print("Processing file:", jsonl_file)

    data_df = pd.read_json(jsonl_file, lines=True)
    # drop empty chunck
    data_df = data_df[data_df["chunk"].apply(lambda x: isinstance(x, str) and x.strip() != "")]

    print("Shape:", data_df.shape)
    print(data_df.head())

    chunks = data_df["chunk"].values
    chunks = chunks.tolist()

    embeddings = generate_text_embeddings(
        chunks, EMBEDDING_DIMENSION, batch_size=15)
    data_df["embedding"] = embeddings

    time.sleep(5)

    # Save
    print("Shape:", data_df.shape)
    print(data_df.head())

    jsonl_filename = jsonl_file.replace("semantic chunks-", "embeddings-")
    with open(jsonl_filename, "w") as json_file:
        json_file.write(data_df.to_json(orient='records', lines=True))

Number of files to process: 1
Processing file: outputs\semantic chunks-hpi-sec-sser.pdf.jsonl
Shape: (469, 2)
                                               chunk              book
0  1\nHPI SEC & \nSSER\nPatient safety measuremen...  hpi-sec-sser.pdf
1  While healthcare holds healing without \nharm ...  hpi-sec-sser.pdf
2  It serves as the foundation for \nthe calculat...  hpi-sec-sser.pdf
3  •\t\nFocus on other frequently \nencountered i...  hpi-sec-sser.pdf
4  The application of these concepts has \nbeen e...  hpi-sec-sser.pdf
Shape: (469, 3)
                                               chunk              book  \
0  1\nHPI SEC & \nSSER\nPatient safety measuremen...  hpi-sec-sser.pdf   
1  While healthcare holds healing without \nharm ...  hpi-sec-sser.pdf   
2  It serves as the foundation for \nthe calculat...  hpi-sec-sser.pdf   
3  •\t\nFocus on other frequently \nencountered i...  hpi-sec-sser.pdf   
4  The application of these concepts has \nbeen e...  hpi-sec-sser.pdf   

   

In [25]:
import json
emb_file_name = chunk_file.replace("semantic chunks-", "embeddings-")
input_file = os.path.join(OUTPUT_FOLDER, emb_file_name)
output_file_name = f"vertex-ready-{emb_file_name}"
output_file = f"outputs/{output_file_name}"

records = []
with open(input_file, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        obj = json.loads(line)
        record = {
            "id": str(i),
            "embedding": obj["embedding"],
            "data": obj["chunk"]  # or f"{obj['book']}: {obj['chunk']}"
        }
        records.append(record)

with open(output_file, "w", encoding="utf-8") as f:
    for r in records:
        f.write(json.dumps(r) + "\n")

print(f"✅ Reformatted file saved: {output_file}")

✅ Reformatted file saved: outputs/vertex-ready-embeddings-hpi-sec-sser.pdf.jsonl


In [26]:
destination_blob_name = f"Embedding/{output_file_name}"
blob = bucket.blob(destination_blob_name)
blob.upload_from_filename(output_file)
print(f"✅ Uploaded to gs://{bucket_name}/{destination_blob_name}")


✅ Uploaded to gs://88-data/Embedding/vertex-ready-embeddings-hpi-sec-sser.pdf.jsonl


In [27]:
from google.cloud import aiplatform
GCP_LOCATION = "us-central1"
index_display_name = "book-embeddings-index"
aiplatform.init(project=gcp_project, location=GCP_LOCATION)

index = aiplatform.MatchingEngineIndex.create_tree_ah_index(
    display_name=index_display_name,
    contents_delta_uri=f"gs://{bucket_name}/Embedding/{output_file_name}",
    dimensions=256,   
    approximate_neighbors_count=10,
)

print("✅ Created index:", index.resource_name)

Creating MatchingEngineIndex
Create MatchingEngineIndex backing LRO: projects/748223712605/locations/us-central1/indexes/3074254302470995968/operations/1958131613799809024
MatchingEngineIndex created. Resource name: projects/748223712605/locations/us-central1/indexes/3074254302470995968
To use this MatchingEngineIndex in another session:
index = aiplatform.MatchingEngineIndex('projects/748223712605/locations/us-central1/indexes/3074254302470995968')
✅ Created index: projects/748223712605/locations/us-central1/indexes/3074254302470995968


In [28]:
endpoint = aiplatform.MatchingEngineIndexEndpoint.create(
    display_name="book-rag-endpoint",
    public_endpoint_enabled=True,
)

Creating MatchingEngineIndexEndpoint
Create MatchingEngineIndexEndpoint backing LRO: projects/748223712605/locations/us-central1/indexEndpoints/4030108139008294912/operations/3917197451705974784
MatchingEngineIndexEndpoint created. Resource name: projects/748223712605/locations/us-central1/indexEndpoints/4030108139008294912
To use this MatchingEngineIndexEndpoint in another session:
index_endpoint = aiplatform.MatchingEngineIndexEndpoint('projects/748223712605/locations/us-central1/indexEndpoints/4030108139008294912')


In [30]:
endpoint.deploy_index(
    index=index,
    deployed_index_id="book_RAG_index" 
)

Deploying index MatchingEngineIndexEndpoint index_endpoint: projects/748223712605/locations/us-central1/indexEndpoints/4030108139008294912
Deploy index MatchingEngineIndexEndpoint index_endpoint backing LRO: projects/748223712605/locations/us-central1/indexEndpoints/4030108139008294912/operations/3146518965472198656
MatchingEngineIndexEndpoint index_endpoint Deployed index. Resource name: projects/748223712605/locations/us-central1/indexEndpoints/4030108139008294912


resource name: projects/748223712605/locations/us-central1/indexEndpoints/4030108139008294912

In [31]:
aiplatform.init(project="apcomp215-group88", location="us-central1")

# Connect to your endpoint
endpoint = aiplatform.MatchingEngineIndexEndpoint(
    "projects/apcomp215-group88/locations/us-central1/indexEndpoints/4030108139008294912"
)

# Your sample query embedding (e.g., from Gemini or same embedding model)
query_vector = [0.1, 0.2]  # replace with actual embedding

response = endpoint.find_neighbors(
    deployed_index_id="book_RAG_index",
    queries=[query_vector],
    num_neighbors=5,
)

for match in response[0].neighbors:
    print("Similarity:", match.distance)
    print("Matched text:", match.datapoint.datapoint_id)

IndexError: list index out of range

In [32]:
def generate_query_embedding(query):
    kwargs = {
        "output_dimensionality": EMBEDDING_DIMENSION
    }
    response = llm_client.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=query,
        config=types.EmbedContentConfig(**kwargs)
    )
    return response.embeddings[0].values


In [33]:
EMBEDDING_DIMENSION = 256
from google.genai import types
query = "HPI Safety Event Classification levels"
query_embedding = generate_query_embedding(query)
print("Embedding values:", query_embedding)


response = endpoint.find_neighbors(
    deployed_index_id="book_RAG_index",
    queries=[query_embedding],
    num_neighbors=5,
)
print("Raw response:", response)
print("Response type:", type(response))
print("Length:", len(response))
for match in response[0].neighbors:
    print("Similarity:", match.distance)
    print("Matched text:", match.datapoint.datapoint_id)

Embedding values: [0.06678880751132965, 0.01975920982658863, -0.025162428617477417, -0.023826275020837784, 0.003035306930541992, 0.045707203447818756, 0.049287162721157074, 0.023671753704547882, -0.03245343640446663, -0.004454673733562231, 0.015579811297357082, 0.04972235485911369, -0.008825595490634441, 0.07507538795471191, -0.015258429571986198, -0.02658715844154358, 0.03610214591026306, 0.022252120077610016, -0.07225276529788971, -0.04432602971792221, -0.056875213980674744, 0.006762172095477581, 0.027221057564020157, -0.03583698719739914, -0.005424549803137779, -0.04565829411149025, 0.0009299803641624749, 0.00871178600937128, -0.04505099728703499, -0.006529178470373154, 0.030019616708159447, 0.0446130707859993, 0.02358684130012989, 0.038112882524728775, 0.036166153848171234, -0.02548186108469963, -0.009342627599835396, 0.021406134590506554, 0.02998342737555504, -0.026867400854825974, -0.010987815447151661, -0.03683635964989662, 0.01745135337114334, 0.05263826996088028, -0.0782771334

IndexError: list index out of range